In [1]:
import os, json, re, time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

DEV_PATH   = "project/data/dev.jsonl"
TEST_PATH  = "project/data/test_30.jsonl"
TRAIN_PATH = "project/data/train.jsonl"

RUN_TEST = True  # True só no fim

SPLITS = {
    "dev": DEV_PATH,
    **({"test": TEST_PATH} if RUN_TEST else {})
}

Path("outputs").mkdir(exist_ok=True, parents=True)
print("Setup OK | splits:", list(SPLITS.keys()))


Setup OK | splits: ['dev', 'test']


In [3]:
import json

TRAIN_PATH = "project/data/train.jsonl"  # ou o path correto no teu notebook

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train = [json.loads(line) for line in f if line.strip()]

print("Train loaded:", len(train))


Train loaded: 310


In [4]:
# train já carregado como lista de dicts (jsonl)
SHOT_IDXS = [266, 38, 274]  # podes ajustar; 266 é a pergunta "limpa"

shots = [train[i] for i in SHOT_IDXS]

examples_str = "\n".join(
    json.dumps({"question": s["instruction"], "answer": s["response"]}, ensure_ascii=False)
    for s in shots
)

print("Few-shots escolhidos:")
for i, s in zip(SHOT_IDXS, shots):
    print(f"[{i}] {s['instruction']}")



Few-shots escolhidos:
[266] Do GLP-1 receptor agonists mimic caloric restriction?
[38] If evidence is limited, say so. Do GLP-1 receptor agonists mimic caloric restriction?
[274] Answer conservatively and include safety notes if relevant: Do GLP-1 receptor agonists mimic caloric restriction?


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
model.eval()

gen_cfg = GenerationConfig(
    max_new_tokens=260,   # few-shot precisa mais que 180
    do_sample=False,
    temperature=0.0,
    use_cache=True,
)

print("Model loaded.")
print("CUDA available:", torch.cuda.is_available())



`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-19 13:20:48.103157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768828848.126223     569 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768828848.135828     569 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-19 13:20:48.304670: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded.
CUDA available: True


In [6]:
SYSTEM_MSG = (
    "You are a cautious healthcare assistant. "
    "Return ONLY a valid JSON object with keys: short_answer, confidence_level, clinical_notes. "
    "confidence_level must be one of: high, medium, low. "
    "No markdown, no code fences, no extra text."
)

def load_questions(path: str):
    qs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            q = obj.get("question") or obj.get("instruction")
            if q and isinstance(q, str):
                qs.append(q.strip())
    return qs

def extract_any_json(text: str):
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        depth = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        break
    return None



In [7]:
def generate_few_shot(question: str):
    user_msg = (
        "Here are examples of the required format:\n"
        f"{examples_str}\n\n"
        "Now answer the next question in the same format.\n"
        f"Question: {question}\n"
        "JSON:"
    )

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_msg},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # NÃO forces '{' aqui (evita outputs “estranhos” e não resolve truncamento)
    # prompt = prompt + "{"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            generation_config=gen_cfg,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    dt = time.time() - t0

    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    gen_text = full_text[len(prompt_text):].strip()

    # FIX: às vezes vem sem "{"
    if gen_text.lstrip().startswith('"'):
        gen_text = "{" + gen_text

    obj = extract_any_json(gen_text)
    if obj is None:
        obj = {
            "short_answer": "",
            "confidence_level": "low",
            "clinical_notes": "Model output did not contain valid JSON."
        }

    return obj, gen_text, dt


In [8]:
def run_split(split_name: str, in_path: str, method_name: str = "few"):
    questions = load_questions(in_path)
    n = len(questions)
    if n == 0:
        raise ValueError(f"Não carreguei perguntas de {in_path}")

    out_path = f"outputs/{split_name}_few_shot.jsonl"
    print(f"\n=== RUN FEW | split={split_name} | n={n} ===")
    print("Input:", in_path)
    print("Output:", out_path)

    t_global0 = time.time()
    times = []

    with open(out_path, "w", encoding="utf-8") as f:
        for i, q in enumerate(questions, start=1):
            t_start = time.time()
            print(f"[{split_name}] {i}/{n} → start {time.strftime('%H:%M:%S')}", flush=True)

            obj, raw_text, dt = generate_few_shot(q)
            times.append(dt)

            avg = sum(times) / len(times)
            eta = avg * (n - i)

            row = {
                "split": split_name,
                "method": method_name,
                "question": q,
                "parsed_json": obj,
                "raw_text": raw_text,
                "timing_sec": dt
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()

            print(
                f"[{split_name}] {i}/{n} ← done {time.time()-t_start:.1f}s | "
                f"avg {avg:.1f}s | ETA {eta/60:.1f} min",
                flush=True
            )

    total = time.time() - t_global0
    avg_time = (sum(times)/len(times)) if times else float("nan")
    print(f"Saved: {out_path} | total {total/60:.1f} min | avg {avg_time:.2f}s/question")
    return out_path

generated_files = []
for split_name, in_path in SPLITS.items():
    generated_files.append(run_split(split_name, in_path, method_name="few"))




=== RUN FEW | split=dev | n=30 ===
Input: project/data/dev.jsonl
Output: outputs/dev_few_shot.jsonl
[dev] 1/30 → start 13:22:48
[dev] 1/30 ← done 3.0s | avg 3.0s | ETA 1.5 min
[dev] 2/30 → start 13:22:51
[dev] 2/30 ← done 2.9s | avg 3.0s | ETA 1.4 min
[dev] 3/30 → start 13:22:54
[dev] 3/30 ← done 2.6s | avg 2.8s | ETA 1.3 min
[dev] 4/30 → start 13:22:56
[dev] 4/30 ← done 2.6s | avg 2.8s | ETA 1.2 min
[dev] 5/30 → start 13:22:59
[dev] 5/30 ← done 2.6s | avg 2.7s | ETA 1.1 min
[dev] 6/30 → start 13:23:01
[dev] 6/30 ← done 3.1s | avg 2.8s | ETA 1.1 min
[dev] 7/30 → start 13:23:04
[dev] 7/30 ← done 2.9s | avg 2.8s | ETA 1.1 min
[dev] 8/30 → start 13:23:07
[dev] 8/30 ← done 3.1s | avg 2.8s | ETA 1.0 min
[dev] 9/30 → start 13:23:10
[dev] 9/30 ← done 2.4s | avg 2.8s | ETA 1.0 min
[dev] 10/30 → start 13:23:13
[dev] 10/30 ← done 4.0s | avg 2.9s | ETA 1.0 min
[dev] 11/30 → start 13:23:17
[dev] 11/30 ← done 2.2s | avg 2.9s | ETA 0.9 min
[dev] 12/30 → start 13:23:19
[dev] 12/30 ← done 4.6s | avg 

In [9]:
import json, random

required = {"short_answer", "confidence_level", "clinical_notes"}
allowed_conf = {"high","medium","low"}

for path in generated_files:
    rows = [json.loads(l) for l in open(path, "r", encoding="utf-8") if l.strip()]
    missing = bad = empty = 0
    for r in rows:
        pj = r.get("parsed_json", {})
        if not isinstance(pj, dict) or not required.issubset(pj.keys()):
            missing += 1
            continue
        if str(pj.get("confidence_level","")).lower() not in allowed_conf:
            bad += 1
        if not str(pj.get("short_answer","")).strip():
            empty += 1

    print("\nFILE:", path, "| lines:", len(rows))
    print("Missing keys:", missing)
    print("Bad confidence:", bad)
    print("Empty short_answer:", empty)

    s = random.choice(rows)
    print("Sample Q:", s["question"])
    print("Sample parsed_json:", s["parsed_json"])



FILE: outputs/dev_few_shot.jsonl | lines: 30
Missing keys: 0
Bad confidence: 0
Empty short_answer: 0
Sample Q: In clinical terms, What is cellular senescence in aging biology?
Sample parsed_json: {'short_answer': 'Cellular senescence is a state in which cells cease to divide and typically secrete pro-inflammatory factors, contributing to age-related tissue dysfunction and pathology.', 'confidence_level': 'high', 'clinical_notes': 'This definition is widely accepted in the field of aging biology and is supported by extensive research.'}

FILE: outputs/test_few_shot.jsonl | lines: 30
Missing keys: 0
Bad confidence: 0
Empty short_answer: 0
Sample Q: What are common misconceptions about anti-aging drugs?
Sample parsed_json: {'short_answer': 'Common misconceptions about anti-aging drugs include the belief that they can reverse aging or significantly extend lifespan, when in reality they primarily target specific aspects of aging such as inflammation or cellular senescence, and their effect

In [10]:
import boto3
from pathlib import Path

S3_BUCKET = "healthcare-longevity"
S3_PREFIX = "healthcare-longevity"

s3 = boto3.client("s3")

print("S3 client ready")

OUT_PATH = "outputs/dev_few_shot.jsonl"  # ajusta se necessário
key = f"{S3_PREFIX}/results/dev_few_shot.jsonl"

s3.upload_file(OUT_PATH, S3_BUCKET, key)
print(f"Uploaded to s3://{S3_BUCKET}/{key}")


S3 client ready
Uploaded to s3://healthcare-longevity/healthcare-longevity/results/dev_few_shot.jsonl


In [11]:
def upload_if_exists(local_path: str, s3_key: str):
    p = Path(local_path)
    if not p.exists():
        print("SKIP (not found):", local_path)
        return
    s3.upload_file(str(p), S3_BUCKET, s3_key)
    print(f"UPLOADED → s3://{S3_BUCKET}/{s3_key}")

print("\n=== UPLOADING FEW-SHOT OUTPUTS ===")
upload_if_exists("outputs/dev_few_shot.jsonl",  f"{S3_PREFIX}/results/dev_few_shot.jsonl")
upload_if_exists("outputs/test_few_shot.jsonl", f"{S3_PREFIX}/results/test_few_shot.jsonl")

print("\n=== UPLOADING DATASETS (optional but recommended) ===")
upload_if_exists("project/data/train-2.jsonl",  f"{S3_PREFIX}/data/train-2.jsonl")
upload_if_exists("project/data/dev_questions.jsonl", f"{S3_PREFIX}/data/dev_questions.jsonl")
upload_if_exists("project/data/test_30.jsonl",  f"{S3_PREFIX}/data/test_30.jsonl")

print("\nDone.")



=== UPLOADING FEW-SHOT OUTPUTS ===
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/dev_few_shot.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/results/test_few_shot.jsonl

=== UPLOADING DATASETS (optional but recommended) ===
SKIP (not found): project/data/train-2.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/data/dev_questions.jsonl
UPLOADED → s3://healthcare-longevity/healthcare-longevity/data/test_30.jsonl

Done.


In [12]:
def fix_file(in_path: str, out_path: str):
    inp = Path(in_path)
    if not inp.exists():
        print("SKIP (not found):", in_path)
        return

    rows = [json.loads(l) for l in inp.open("r", encoding="utf-8") if l.strip()]
    fixed = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for r in rows:
            text = (r.get("raw_text") or "").strip()
            if text.startswith('"'):
                text = "{" + text

            obj = extract_any_json(text)
            if obj is not None:
                r["parsed_json"] = obj
                fixed += 1

            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Wrote: {out_path} | fixed {fixed}/{len(rows)}")

fix_file("outputs/dev_few_shot.jsonl",  "outputs/dev_few_shot_fixed.jsonl")
fix_file("outputs/test_few_shot.jsonl", "outputs/test_few_shot_fixed.jsonl")


Wrote: outputs/dev_few_shot_fixed.jsonl | fixed 30/30
Wrote: outputs/test_few_shot_fixed.jsonl | fixed 30/30
